<a href="https://colab.research.google.com/github/majorluvale/hpc-data-analysis/blob/main/Analyse_HPC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [45]:
import json
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as s
import warnings
warnings.filterwarnings("ignore")

#Sklearn
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier
from sklearn.metrics import (accuracy_score,precision_score, f1_score, confusion_matrix, classification_report, roc_auc_score, roc_curve)
from sklearn.preprocessing import StandardScaler

#Configuration des graphes
plt.style.use("seaborn-v0_8-darkgrid")
s.set_palette("husl")
%matplotlib inline

#Reproductivité
np.random.seed(42)

In [3]:
PLAN_YEAR = 2025 #@param {type:"slider", min:2021, max:2025, step:1}

In [4]:
def getPlanIds(year):
    resp = requests.get(f'https://api.hpc.tools/v2/public/plan?year={year}')
    planIds = [plan['id'] for plan in resp.json()['data']]
    return planIds

In [5]:
len(getPlanIds(2026))

0

In [6]:
url = 'https://api.hpc.tools/v2/public/planSummary?year=2025&includeIndicators=true&includeCaseloads=true&includeFinancials=true&includeDisaggregatedData=True'
data = requests.get(url).json()['data']['planData']

In [7]:
#Normaliser les data. L'API renvoit du json
df = pd.json_normalize(data,
                       record_path=['caseloads', 'measurements'],
                       meta=['planId', 'name', 'shortName', 'subtitle', 'planLanguage', 'isReleased', 'planClusterType', 'planType', 'planCostingType', 'planYear', 'lastPublishedDate', 'planStartDate', 'planEndDate',
                        ['caseloads', 'inNeed'], ['caseloads', 'target'], ['caseloads', 'availableGlobalClusterCode'], ['caseloads', 'caseloadDescription']])
df.head()

,monitoringPeriodId,periodicalReach,cumulativeReach,covered,optionOverallPeriodicalReach,optionOverallCumulReach,planId,name,shortName,subtitle,...,planType,planCostingType,planYear,lastPublishedDate,planStartDate,planEndDate,caseloads.inNeed,caseloads.target,caseloads.availableGlobalClusterCode,caseloads.caseloadDescription
0,3832,192956,404811,None,NaN,NaN,1266,Cameroon Humanitarian Response Plan 2025,Cameroon,Humanitarian Response Plan,...,Humanitarian response plan,Projects costed requirements,2025,24/09/2025,01/01/2025,31/12/2025,3322058,2100146,n/a,Final HRP caseload
1,3832,None,None,None,NaN,NaN,1266,Cameroon Humanitarian Response Plan 2025,Cameroon,Humanitarian Response Plan,...,Humanitarian response plan,Projects costed requirements,2025,24/09/2025,01/01/2025,31/12/2025,None,None,CSS,Coordination and Support Services
2,3832,41459,141304,None,NaN,NaN,1266,Cameroon Humanitarian Response Plan 2025,Cameroon,Humanitarian Response Plan,...,Humanitarian response plan,Projects costed requirements,2025,24/09/2025,01/01/2025,31/12/2025,1496550,1126009,EDU,Education
3,3832,88542,179520,None,NaN,NaN,1266,Cameroon Humanitarian Response Plan 2025,Cameroon,Humanitarian Response Plan,...,Humanitarian response plan,Projects costed requirements,2025,24/09/2025,01/01/2025,31/12/2025,2483832,1052751,FSC,Food Security
4,3832,87701,187933,None,NaN,NaN,1266,Cameroon Humanitarian Response Plan 2025,Cameroon,Humanitarian Response Plan,...,Humanitarian response plan,Projects costed requirements,2025,24/09/2025,01/01/2025,31/12/2025,1511471,1090263,HEA,Health


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 444 entries, 0 to 443
Data columns (total 23 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   monitoringPeriodId                    444 non-null    int64  
 1   periodicalReach                       290 non-null    object 
 2   cumulativeReach                       326 non-null    object 
 3   covered                               64 non-null     object 
 4   optionOverallPeriodicalReach          13 non-null     object 
 5   optionOverallCumulReach               13 non-null     float64
 6   planId                                444 non-null    object 
 7   name                                  444 non-null    object 
 8   shortName                             444 non-null    object 
 9   subtitle                              444 non-null    object 
 10  planLanguage                          444 non-null    object 
 11  isReleased         

In [9]:
#Supprimer les colonnes dont on aura pas besoin
df.drop(columns=['monitoringPeriodId', 'periodicalReach', 'covered',
       'optionOverallCumulReach','optionOverallPeriodicalReach','planLanguage',
       'name','subtitle',
       'caseloads.caseloadDescription'], inplace=True)

In [10]:
df.rename(columns={'cumulativeReach':'Reach', 'shortName':'planName','planYear':'Year',
       'caseloads.inNeed':'inNeed','caseloads.target':'Target', 'caseloads.availableGlobalClusterCode':'GlobalClusterCode'}, inplace=True)

#Réorganiser les colonnes
df = df[['planId', 'planName', 'inNeed', 'Target', 'Reach', 'Year', 'planClusterType', 'planType', 'lastPublishedDate','planCostingType', 'planStartDate', 'planEndDate', 'isReleased', 'GlobalClusterCode']]

In [12]:
#Null values
df.isna().sum()

planId                 0
planName               0
inNeed                45
Target                13
Reach                118
Year                   0
planClusterType        0
planType               0
lastPublishedDate      0
planCostingType        0
planStartDate          0
planEndDate            0
isReleased             0
GlobalClusterCode      0
dtype: int64

In [14]:
pd.set_option('future.no_silent_downcasting', True)
#Remplacer les nulls par 0
df.fillna(0, inplace=True)

In [20]:
#Remplacer les valeurs nulles par 0
df['inNeed'] = df['inNeed'].replace('',0)
df['Target'] = df['Target'].replace('',0)
df['Reach'] = df['Reach'].replace('',0)
df['Year'] = df['Year'].replace('',0)
df['planId'] = df['planId'].replace('',0)

In [22]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 444 entries, 0 to 443
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   planId             444 non-null    object
 1   planName           444 non-null    object
 2   inNeed             444 non-null    object
 3   Target             444 non-null    int64 
 4   Reach              444 non-null    object
 5   Year               444 non-null    object
 6   planClusterType    444 non-null    object
 7   planType           444 non-null    object
 8   lastPublishedDate  444 non-null    object
 9   planCostingType    444 non-null    object
 10  planStartDate      444 non-null    object
 11  planEndDate        444 non-null    object
 12  isReleased         444 non-null    object
 13  GlobalClusterCode  444 non-null    object
dtypes: int64(1), object(13)
memory usage: 48.7+ KB


In [38]:
df['Reach'].astype('float')

0       404811.0
1            0.0
2       141304.0
3       179520.0
4       187933.0
         ...    
439     242042.0
440     588542.0
441     213384.0
442     958316.0
443    1353611.0
Name: Reach, Length: 444, dtype: float64

In [36]:
df['Reach'].str.strip().isna().sum()

357

In [40]:

#Convertir planId, inNeed, Target, Reach, Year en integer.
df['inNeed'] = df['inNeed'].astype('int')
df['Reach'] = df['Reach'].astype('float')
df['Reach'] = df['Reach'].astype('int')
df['Year'] = df['Year'].astype('int')          
df['Target'] = df['Target'].astype('int')    
#df['lastPublishedDate'] = df['lastPublishedDate'].astype('date32[pyarrow]')     


In [51]:
#check for null values
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 444 entries, 0 to 443
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   planId             444 non-null    object
 1   planName           444 non-null    object
 2   inNeed             444 non-null    int32 
 3   Target             444 non-null    int32 
 4   Reach              444 non-null    int32 
 5   Year               444 non-null    int32 
 6   planClusterType    444 non-null    object
 7   planType           444 non-null    object
 8   lastPublishedDate  444 non-null    object
 9   planCostingType    444 non-null    object
 10  planStartDate      444 non-null    object
 11  planEndDate        444 non-null    object
 12  isReleased         444 non-null    object
 13  GlobalClusterCode  444 non-null    object
dtypes: int32(4), object(10)
memory usage: 41.8+ KB


In [ ]:
#remplacer tous les nulls par 0
df.fillna(0, inplace=True)
df.isna().sum()

planId               0
planName             0
inNeed               0
Target               0
Reach                0
Year                 0
planClusterType      0
planType             0
lastPublishedDate    0
planCostingType      0
planStartDate        0
planEndDate          0
isReleased           0
GlobalClusterCode    0
dtype: int64

In [50]:
df.head()

,planId,planName,inNeed,Target,Reach,Year,planClusterType,planType,lastPublishedDate,planCostingType,planStartDate,planEndDate,isReleased,GlobalClusterCode
0,1266,Cameroon,3322058,2100146,404811,2025,cluster,Humanitarian response plan,24/09/2025,Projects costed requirements,01/01/2025,31/12/2025,True,n/a
1,1266,Cameroon,0,0,0,2025,cluster,Humanitarian response plan,24/09/2025,Projects costed requirements,01/01/2025,31/12/2025,True,CSS
2,1266,Cameroon,1496550,1126009,141304,2025,cluster,Humanitarian response plan,24/09/2025,Projects costed requirements,01/01/2025,31/12/2025,True,EDU
3,1266,Cameroon,2483832,1052751,179520,2025,cluster,Humanitarian response plan,24/09/2025,Projects costed requirements,01/01/2025,31/12/2025,True,FSC
4,1266,Cameroon,1511471,1090263,187933,2025,cluster,Humanitarian response plan,24/09/2025,Projects costed requirements,01/01/2025,31/12/2025,True,HEA


In [44]:
df.values

array([[1266, 'Cameroon', 3322058, ..., '31/12/2025', True, 'n/a'],
       [1266, 'Cameroon', 0, ..., '31/12/2025', True, 'CSS'],
       [1266, 'Cameroon', 1496550, ..., '31/12/2025', True, 'EDU'],
       ...,
       [1275, 'Myanmar (Original)', 7304649, ..., '31/12/2025', True,
        'PRO-MIN'],
       [1275, 'Myanmar (Original)', 5118207, ..., '31/12/2025', True,
        'SHL'],
       [1275, 'Myanmar (Original)', 6854169, ..., '31/12/2025', True,
        'WSH']], dtype=object)

In [47]:
num_data = df.select_dtypes('number')
num_data.corr()

,inNeed,Target,Reach,Year
inNeed,1.000000,0.862903,0.651699,NaN
Target,0.862903,1.000000,0.789616,NaN
Reach,0.651699,0.789616,1.000000,NaN
Year,NaN,NaN,NaN,NaN
